# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is fully FAIR-compliant. All aspects of the dataset are referenced by their Croissant `@id` fields.

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata and available records from the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (as an object)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review record sets, fields, and their Croissant `@id`s. This helps you understand how to reference data programmatically for loading and analysis.

In [ ]:
# List all record sets by their @id and print their fields and columns (@id)

for record_set in dataset.record_sets:
    print(f"\nRecordSet: {record_set.id}")
    if hasattr(record_set, 'fields') and record_set.fields:
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - {field.id} (type: {getattr(field, 'data_type', None)})")
    if hasattr(record_set, 'columns') and record_set.columns:
        print("  Columns:")
        for column in record_set.columns:
            print(f"    - {column.id}")

## 3. Data Extraction
Extract tabular records from each record set by referencing their `@id` fields. Each record set is loaded into a pandas DataFrame using the `dataset.records()` API.

In [ ]:
# Identify all record_set @ids for extraction
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Record Sets in dataset (by @id):")
for rid in record_set_ids:
    print(f"  - {rid}")

dataframes = {}
# Extract each record set into a DataFrame if records are available
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set '{record_set_id}' with shape {dataframes[record_set_id].shape}")

# Show columns and first few rows for a selected record set (if any loaded, pick the first)
if dataframes:
    example_record_set_id = next(iter(dataframes.keys()))
    print(f"\nColumns for Record Set '{example_record_set_id}':")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No tabular records found in available record sets.")

## 4. Exploratory Data Analysis (EDA)
Process fields from the most populated record set: filter, normalize, and group by key attributes. All references are made using their corresponding `@id`.

_**Adjust variable names and field @ids below based on actual field IDs printed from above.**_


In [ ]:
# Pick a record set (by @id) and select a numeric field for analysis
# -- replace these with the actual found record_set and field @ids for your dataset!
if dataframes:
    record_set_id = example_record_set_id  # e.g., 'cr:RecordSet1'
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}")

    # Attempt to auto-detect a numeric field (@id)
    numeric_field_id = None
    for col in df.columns:
        # Check if column appears numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if not numeric_field_id:
        print("No numeric fields found for processing. EDA steps will be skipped.")
    else:
        print(f"Numeric field detected (@id): {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        print(f"Threshold set to mean: {threshold}")

        # Filter records
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}: {filtered_df.shape[0]} rows shown below")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} (added column '{norm_col}'):")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt to group by a non-numeric categorical field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == 'object':
                group_field_id = col
                break
        if group_field_id:
            print(f"Grouping by field @id: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            display(grouped_df.head())
        else:
            print("No categorical field available for grouping.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize distributions of fields or relationships between variables in your tabular record set. All axes/legends use Croissant `@id`s for clarity.

In [ ]:
import matplotlib.pyplot as plt

if dataframes and numeric_field_id:
    plt.figure(figsize=(7,4))
    df[numeric_field_id].hist(bins=30, alpha=0.7)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field available
    if group_field_id:
        plt.figure(figsize=(8,4))
        grouped_df.plot.bar(y=f"mean_{numeric_field_id}", legend=False)
        plt.title(f"Mean of '{numeric_field_id}' grouped by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("Visualization skipped: No numeric data available.")

## 6. Conclusion
This notebook demonstrated how to programmatically explore a FAIR Croissant dataset using only record set/field/column `@id`s. You:
- Loaded metadata and records from the Croissant schema.
- Inspected the available structure (record sets, fields, columns).
- Extracted data using only `@id` references.
- Filtered, normalized, grouped, and visualized data for basic EDA.

To dive deeper, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/) and use specific `@id`s found within your dataset.